# Prompt Evaluation — Part 3: Add Deterministic Graders

A model judge is flexible but subjective. When part of "correct" is objectively checkable, **check it with code, not a model** — code graders are free, instant, and 100% consistent. This notebook adds them.

**What's new:**

- **Code-based validators** (`validate_json` / `validate_python` / `validate_regex`) — each just tries to parse the output and returns 10 or 0. No model call, no ambiguity: the code either compiles or it doesn't.
- **Two graders combined into one score:** `score = (model_score + syntax_score) / 2`. The deterministic check anchors the fuzzy model judgment. A verbose answer can no longer score well if it isn't even valid code.
- **`run_prompt` now constrains the output** ("Respond only with Python, JSON, or a plain Regex — no commentary") and prefills ` ```code `. This is what makes the output *mechanically gradable* — you can't run `ast.parse` on an answer wrapped in three paragraphs of explanation. **Design the prompt and the grader together.**

**General principle:** prefer the cheapest grader that captures what you care about — exact match / code check > model judge > human review. Reach for the model only for the parts a simple check can't capture.

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

**Setup — unchanged from notebook 002.** Same client, same cheap model.

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

**Helpers — unchanged.** The `stop_sequences` lever gets leaned on harder here, in `run_prompt`.

In [3]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

**Dataset generator — unchanged** (`{task, format}`). The `format` field starts earning its keep below: it tells the syntax grader whether to parse the output as JSON, Python, or Regex.

In [4]:
# dataset.json is generated once (notebook 002) and reused here so scores stay comparable.
# The check regenerates it only if it's missing, so a from-scratch run of this notebook still works.
import os

if os.path.exists("dataset.json"):
    print("dataset.json already exists — reusing it (skipping generation). Delete the file to regenerate.")
else:
    dataset = generate_dataset()
    with open("dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)
    print("dataset.json not found — generated a fresh one.")

dataset.json already exists — reusing it (skipping generation). Delete the file to regenerate.


**Reuse, don't regenerate.** The check keeps the same `dataset.json` from notebook 002 so the scores here are directly comparable — you're measuring the effect of the new graders, not a different set of tasks.

In [5]:
# Function to grade a test case + output using a model
import re


def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])

    # LLM-authored JSON is fragile: when the judge echoes the solution's regex or Python,
    # its raw backslashes (\d, \., \w ...) are illegal JSON escapes and json.loads raises
    # "Invalid \escape". If the first parse fails, escape any backslash that isn't a valid
    # JSON escape and try once more.
    try:
        return json.loads(eval_text)
    except json.JSONDecodeError:
        repaired = re.sub(r'\\(?!["\\/bfnrtu])', r"\\\\", eval_text)
        return json.loads(repaired)

**Model judge — same prompt, but the parsing is now hardened.** The judge is still subjective (scores cluster around 7–8); the code graders two cells down anchor it. The one real change: when the graded solution is a **regex or Python**, the judge echoes its raw backslashes (`\d`, `\.`, `\w`) into the JSON — illegal escapes that make `json.loads` throw `Invalid \escape`. The `try/except` repairs those and re-parses. **Lesson: never trust LLM-authored JSON to parse on the first try — degrade gracefully.**

In [6]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

**Changed: the prompt is now constrained.** "Respond only with Python/JSON/Regex, no commentary" plus the ` ```code ` prefill strips the prose. This isn't cosmetic — bare code is the *only* thing the validators below can actually `parse`. Prompt and grader are designed together.

In [7]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


**New: deterministic graders.** Each validator just tries to `json.loads` / `ast.parse` / `re.compile` the output and returns 10 or 0 — no model call, no ambiguity, free and instant. `grade_syntax` dispatches on the case's `format`. Rule of thumb: **check with code whatever code can check**; save the model judge for the fuzzy parts.

In [8]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

**Changed: hybrid score.** `score = (model_score + syntax_score) / 2`. A fluent but *invalid* answer now gets dragged down by the 0 from the syntax check — verbosity alone can no longer earn a good grade.

In [9]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

**Scoreboard — unchanged.** Same loop-and-average; it just happens to be averaging the new hybrid scores now.

In [10]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.666666666666667


**Run it.** Same dataset as notebook 002, but the average nudges up: constraining the output and gating on valid syntax rewards answers that are genuinely well-formed code. The remaining softness is the judge's — fixed next in notebook 004.

In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\nimport json\n\ndef parse_cloudwatch_log(log_entry):\n    pattern = r'(\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}\\.\\d{3}Z)\\s+(\\w+)\\s+(.*)'\n    match = re.match(pattern, log_entry)\n    \n    if match:\n        return {\n            \"timestamp\": match.group(1),\n            \"log_level\": match.group(2),\n            \"message\": match.group(3)\n        }\n    return None\n\n# Example usage\nlog_entry = \"2024-01-15T10:30:45.123Z ERROR Failed to process request\"\nresult = parse_cloudwatch_log(log_entry)\nprint(json.dumps(result, indent=2))\n",
    "test_case": {
      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message using a regular expression",
      "format": "regex",
      "solution_criteria": "The regex should correctly capture timestamp in ISO 8601 format, log level (INFO, ERROR, WARN, DEBUG), and the remaining message text. Should handle variations in spacing and log formats."
    },
    "score": 8